In [1]:
using LowLevelFEM, LinearAlgebra, Arpack

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 3 for different dependency version already loaded, 4 for different Julia build configuration, 2 for file size changed)
Precompiling packages...
 116.0 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 116 seconds. 106 already precompiled.
[ Info: Precompiling LowLevelFEMArpackExt [402ffc23-fdaa-5e97-8e9c-e0bfb4b9c4b2] (caches not reused: 2 for different dependency version already loaded, 1 for dependency source file not found)
Precompiling packages...
   2.2 s  ✓ LowLevelFEMArpackExt (serial)
  1 dependency successfully precompiled in 4 seconds


In [2]:
structured_box_mesh(lx=100)
#openPreProcessor()

In [3]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [4]:
K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))

sparse([1, 2, 3, 49, 50, 51, 52, 53, 54, 3238  …  360363, 360364, 360365, 360366, 363358, 363359, 363360, 363361, 363362, 363363], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  363363, 363363, 363363, 363363, 363363, 363363, 363363, 363363, 363363, 363363], [4700.8547008547, 1602.5641025641025, -1602.5641025641037, 1068.3760683760697, 801.2820512820518, 320.51282051282175, 1068.3760683760681, -320.51282051282084, -801.2820512820515, -2136.7521367521354  …  -213.6752136743064, 3.68913788406644e-10, 4.172306944383308e-11, 4273.504273504343, -7.144080882426351e-10, -2.2396307031158358e-11, 4273.5042735076, 1.3308181223692372e-9, -1.1823431123048067e-11, 37606.83760683735], 363363, 363363)

In [5]:
f = ∫(U ⋅ [-1, 0, 0], Γ="right");

In [6]:
bc = BoundaryCondition("left", ux=0, uy=0, uz=0);

In [7]:
u = solveField(Symmetric(K), f, support=[bc])
showDoFResults(u, name="u", factor=1);

In [8]:
ε = (u ∘ ∇ + ∇ ∘ u) / 2;

In [9]:
E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1-2ν) * trace(ε) * I);

In [10]:
s = tensorToMatrix(σ);

In [11]:
DG = Matrix{Any}(zeros(9, 9))

DG[1:3, 1:3] = s
DG[4:6, 4:6] = s
DG[7:9, 7:9] = s

Kg = ∫(Grad(U) ⋅ DG ⋅ Grad(U));

In [12]:
u1 = solveEigen(Kg, K, support=[bc])
u1 = LowLevelFEM.mapEigenValues(λ -> -1 / λ, u1);

In [15]:
λ = getEigenValues(u1)

6-element Vector{Float64}:
   4.136369803039813
   4.136369854835233
  37.20314770235539
  37.203147709359584
 103.20805560233437
 103.20805563310894

In [13]:
showDoFResults(u1, name="Buckling shapes");

In [14]:
openPostProcessor();

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
